In [ ]:
# Importe und Pfade
from pathlib import Path
import sys
import platform
import warnings
import pm4py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 160)
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name.lower() == 'notebooks' else CWD
DATA_RAW = PROJECT_ROOT / 'data_raw'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'data_audit'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIG_DIR = OUTPUT_DIR / 'figures'
for folder in [OUTPUT_DIR, TABLE_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('PM4Py:', getattr(pm4py, '__version__', 'version unknown'))
print('pandas:', pd.__version__)
print('Projektwurzel:', PROJECT_ROOT)
print('Data_raw existiert:', DATA_RAW.exists())
print('Output-Ordner:', OUTPUT_DIR)


In [ ]:
# Datensatzpfad prüfen
LOG_PATH = DATA_RAW / 'BPI_Challenge_2018.xes.gz'
if not LOG_PATH.exists():
    candidates = sorted(list(DATA_RAW.glob('*.xes')) + list(DATA_RAW.glob('*.xes.gz')))
    print('Standardpfad nicht gefunden.')
    print('Gefundene Kandidaten:')
    for i, c in enumerate(candidates, start=1):
        print(f'{i}: {c.name}')
    if len(candidates) == 1:
        LOG_PATH = candidates[0]
        print('\nNutze automatisch:', LOG_PATH)
    elif len(candidates) == 0:
        raise FileNotFoundError(f'Keine .xes/.xes.gz-Datei in {DATA_RAW} gefunden.')
    else:
        raise FileNotFoundError('Mehrere Kandidaten gefunden. Setze LOG_PATH manuell auf die richtige Datei.')
print('Log-Pfad:', LOG_PATH)
print('Existiert:', LOG_PATH.exists())
print('Dateigröße in MB:', round(LOG_PATH.stat().st_size / 1024 ** 2, 2))


In [ ]:
# Log laden
raw_log = pm4py.read_xes(str(LOG_PATH))
if isinstance(raw_log, pd.DataFrame):
    df = raw_log.copy()
else:
    df = pm4py.convert_to_dataframe(raw_log)
df['__row_order__'] = np.arange(len(df), dtype=np.int64)
print('Laden abgeschlossen.')
print('Events / Zeilen:', len(df))
print('Spalten:', len(df.columns))
display(df.head())


In [ ]:
# Kernspalten bestimmen
columns = list(df.columns)
print('Alle Spalten:')
for i, col in enumerate(columns, start=1):
    print(f'{i:02d}. {col}')

def find_column(exact_candidates=None, contains_all=None, contains_any=None):
    exact_candidates = exact_candidates or []
    contains_all = contains_all or []
    contains_any = contains_any or []
    lower_cols = {c: c.lower() for c in columns}
    for candidate in exact_candidates:
        if candidate in columns:
            return candidate
    if contains_all:
        for c, lc in lower_cols.items():
            if all((k.lower() in lc for k in contains_all)):
                return c
    if contains_any:
        for c, lc in lower_cols.items():
            if any((k.lower() in lc for k in contains_any)):
                return c
    return None
CASE_COL = find_column(exact_candidates=['case:concept:name', 'case:case', 'case:id', 'case:Application', 'Application'], contains_all=['case', 'concept'])
ACTIVITY_COL = find_column(exact_candidates=['concept:name', 'activity', 'Activity', 'event', 'Event'], contains_any=['concept:name', 'activity'])
TIME_COL = find_column(exact_candidates=['time:timestamp', 'timestamp', 'Timestamp', 'time'], contains_any=['timestamp'])
print('\nAutomatisch erkannte Kernspalten:')
print('CASE_COL    =', CASE_COL)
print('ACTIVITY_COL=', ACTIVITY_COL)
print('TIME_COL    =', TIME_COL)
if CASE_COL is None or ACTIVITY_COL is None or TIME_COL is None:
    raise ValueError('Kernspalten konnten nicht eindeutig erkannt werden. Setze CASE_COL, ACTIVITY_COL, TIME_COL manuell.')


In [ ]:
# Spaltenzuordnung
print('Aktuelle Kernspalten:')
print(CASE_COL, ACTIVITY_COL, TIME_COL)


In [ ]:
# Zeitstempel prüfen
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce', utc=True)
core_missing = pd.DataFrame({'core_column': [CASE_COL, ACTIVITY_COL, TIME_COL], 'missing_count': [df[CASE_COL].isna().sum(), df[ACTIVITY_COL].isna().sum(), df[TIME_COL].isna().sum()], 'missing_pct': [df[CASE_COL].isna().mean() * 100, df[ACTIVITY_COL].isna().mean() * 100, df[TIME_COL].isna().mean() * 100]})
display(core_missing)
core_missing.to_csv(TABLE_DIR / 'core_missing.csv', index=False)
print('Timestamp min:', df[TIME_COL].min())
print('Timestamp max:', df[TIME_COL].max())


In [ ]:
# Speicherfunktionen
def save_table(table: pd.DataFrame, name: str, index: bool=False, show_rows: int | None=20):
    path = TABLE_DIR / f'{name}.csv'
    table.to_csv(path, index=index, encoding='utf-8-sig')
    print(f'Gespeichert: {path}')
    if show_rows is None:
        display(table)
    else:
        display(table.head(show_rows))
    return path

def save_fig(name: str):
    path = FIG_DIR / f'{name}.png'
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches='tight')
    print(f'Gespeichert: {path}')
    plt.show()
    return path

def first_non_null(s):
    s2 = s.dropna()
    return s2.iloc[0] if len(s2) else np.nan

def mode_or_first(s):
    s2 = s.dropna()
    if len(s2) == 0:
        return np.nan
    mode = s2.mode()
    return mode.iloc[0] if len(mode) else s2.iloc[0]


In [ ]:
# Grundübersicht
memory_mb = df.memory_usage(deep=True).sum() / 1024 ** 2
overview = pd.DataFrame([{'log_file': LOG_PATH.name, 'n_events': len(df), 'n_cases': df[CASE_COL].nunique(dropna=True), 'n_activities': df[ACTIVITY_COL].nunique(dropna=True), 'n_columns': len(df.columns), 'timestamp_min': df[TIME_COL].min(), 'timestamp_max': df[TIME_COL].max(), 'memory_mb_dataframe': round(memory_mb, 2), 'full_duplicate_rows': int(df.duplicated().sum())}])
save_table(overview, 'audit_01_overview', show_rows=None)


In [ ]:
# Datentypen und fehlende Werte
missing = pd.DataFrame({'column': df.columns, 'dtype': [str(df[c].dtype) for c in df.columns], 'missing_count': [int(df[c].isna().sum()) for c in df.columns], 'missing_pct': [round(df[c].isna().mean() * 100, 4) for c in df.columns], 'n_unique': [int(df[c].nunique(dropna=True)) for c in df.columns]})
missing = missing.sort_values(['missing_pct', 'n_unique'], ascending=[False, False]).reset_index(drop=True)
save_table(missing, 'audit_02_missing_dtypes_cardinality', show_rows=40)


In [ ]:
# Beispieldaten
display(Markdown('### Erste 10 Events'))
display(df.head(10))
display(Markdown('### Zufallsstichprobe 10 Events'))
display(df.sample(min(10, len(df)), random_state=42))
df_sorted = df.sort_values([CASE_COL, TIME_COL, '__row_order__'], kind='mergesort').reset_index(drop=True)
first_events = df_sorted.groupby(CASE_COL, sort=False).head(1)
last_events = df_sorted.groupby(CASE_COL, sort=False).tail(1)
save_table(first_events[[CASE_COL, ACTIVITY_COL, TIME_COL]].head(50), 'audit_03_first_events_examples', show_rows=20)
save_table(last_events[[CASE_COL, ACTIVITY_COL, TIME_COL]].head(50), 'audit_04_last_events_examples', show_rows=20)


In [ ]:
# Fallmetriken
case_table = df_sorted.groupby(CASE_COL).agg(case_start=(TIME_COL, 'min'), case_end=(TIME_COL, 'max'), n_events=(ACTIVITY_COL, 'size'), n_unique_activities=(ACTIVITY_COL, 'nunique')).reset_index()
case_table['duration_hours'] = (case_table['case_end'] - case_table['case_start']).dt.total_seconds() / 3600
case_table['duration_days'] = case_table['duration_hours'] / 24
case_table['start_calendar_year'] = case_table['case_start'].dt.year
case_table['end_calendar_year'] = case_table['case_end'].dt.year
case_table['rework_intensity_simple'] = case_table['n_events'] - case_table['n_unique_activities']
save_table(case_table, 'audit_05_case_table_basic', show_rows=20)


In [ ]:
# Falllängen und Laufzeiten
def describe_series(s: pd.Series, name: str):
    desc = s.describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame(name).T
    return desc
case_length_desc = describe_series(case_table['n_events'], 'n_events_per_case')
duration_desc = describe_series(case_table['duration_days'], 'duration_days')
save_table(case_length_desc, 'audit_06_case_length_describe', show_rows=None)
save_table(duration_desc, 'audit_07_duration_days_describe', show_rows=None)


In [ ]:
# Falllängen und Laufzeiten abbilden
plt.figure(figsize=(9, 5))
case_table['n_events'].hist(bins=60)
plt.xlabel('Events pro Case')
plt.ylabel('Anzahl Cases')
plt.title('Verteilung der Case Lengths')
save_fig('audit_case_lengths_hist')
cap_len = case_table['n_events'].quantile(0.99)
plt.figure(figsize=(9, 5))
case_table.loc[case_table['n_events'] <= cap_len, 'n_events'].hist(bins=60)
plt.xlabel('Events pro Case bis 99%-Quantil')
plt.ylabel('Anzahl Cases')
plt.title('Case Lengths ohne oberstes 1%')
save_fig('audit_case_lengths_hist_p99')
plt.figure(figsize=(9, 5))
case_table['duration_days'].dropna().hist(bins=60)
plt.xlabel('Durchlaufzeit in Tagen')
plt.ylabel('Anzahl Cases')
plt.title('Verteilung der Case Durations')
save_fig('audit_duration_days_hist')
cap_dur = case_table['duration_days'].quantile(0.99)
plt.figure(figsize=(9, 5))
case_table.loc[case_table['duration_days'] <= cap_dur, 'duration_days'].dropna().hist(bins=60)
plt.xlabel('Durchlaufzeit in Tagen bis 99%-Quantil')
plt.ylabel('Anzahl Cases')
plt.title('Case Durations ohne oberstes 1%')
save_fig('audit_duration_days_hist_p99')


In [ ]:
# Kalenderverteilung
df_sorted['calendar_year'] = df_sorted[TIME_COL].dt.year
df_sorted['calendar_month'] = df_sorted[TIME_COL].dt.to_period('M').astype(str)
events_by_year = df_sorted['calendar_year'].value_counts(dropna=False).sort_index().reset_index()
events_by_year.columns = ['calendar_year', 'n_events']
save_table(events_by_year, 'audit_08_events_by_calendar_year', show_rows=None)
case_starts_by_year = case_table['start_calendar_year'].value_counts(dropna=False).sort_index().reset_index()
case_starts_by_year.columns = ['start_calendar_year', 'n_cases_started']
save_table(case_starts_by_year, 'audit_09_case_starts_by_year', show_rows=None)
case_ends_by_year = case_table['end_calendar_year'].value_counts(dropna=False).sort_index().reset_index()
case_ends_by_year.columns = ['end_calendar_year', 'n_cases_ended']
save_table(case_ends_by_year, 'audit_10_case_ends_by_year', show_rows=None)


In [ ]:
# Ereignisse pro Monat
events_by_month = df_sorted['calendar_month'].value_counts().sort_index().reset_index()
events_by_month.columns = ['calendar_month', 'n_events']
save_table(events_by_month, 'audit_11_events_by_month', show_rows=20)
plt.figure(figsize=(12, 5))
plt.plot(events_by_month['calendar_month'], events_by_month['n_events'], marker='o')
plt.xticks(rotation=90)
plt.xlabel('Kalendermonat')
plt.ylabel('Anzahl Events')
plt.title('Events pro Kalendermonat')
save_fig('audit_events_by_month')


In [ ]:
# Aktivitätshäufigkeiten
activity_counts = df_sorted[ACTIVITY_COL].value_counts(dropna=False).reset_index()
activity_counts.columns = [ACTIVITY_COL, 'n_events']
activity_counts['event_share_pct'] = activity_counts['n_events'] / len(df_sorted) * 100
save_table(activity_counts, 'audit_12_activity_counts', show_rows=50)
plt.figure(figsize=(10, 8))
activity_counts.head(25).sort_values('n_events').plot.barh(x=ACTIVITY_COL, y='n_events', legend=False)
plt.xlabel('Anzahl Events')
plt.ylabel('Activity')
plt.title('Top 25 Activities nach Event-Häufigkeit')
save_fig('audit_top25_activities')


In [ ]:
# Erste und letzte Aktivitäten
first_activity_counts = first_events[ACTIVITY_COL].value_counts(dropna=False).reset_index()
first_activity_counts.columns = ['first_activity', 'n_cases']
first_activity_counts['case_share_pct'] = first_activity_counts['n_cases'] / case_table[CASE_COL].nunique() * 100
save_table(first_activity_counts, 'audit_13_first_activity_counts', show_rows=30)
last_activity_counts = last_events[ACTIVITY_COL].value_counts(dropna=False).reset_index()
last_activity_counts.columns = ['last_activity', 'n_cases']
last_activity_counts['case_share_pct'] = last_activity_counts['n_cases'] / case_table[CASE_COL].nunique() * 100
save_table(last_activity_counts, 'audit_14_last_activity_counts', show_rows=30)


In [ ]:
# Direktfolgebeziehungen
flow_cols = [CASE_COL, ACTIVITY_COL, TIME_COL, '__row_order__']
dfr_df = df_sorted[flow_cols].copy()
dfr_df['next_activity'] = dfr_df.groupby(CASE_COL)[ACTIVITY_COL].shift(-1)
dfr_df['next_timestamp'] = dfr_df.groupby(CASE_COL)[TIME_COL].shift(-1)
dfr_df = dfr_df.dropna(subset=['next_activity']).copy()
dfr_df['delta_hours_to_next'] = (dfr_df['next_timestamp'] - dfr_df[TIME_COL]).dt.total_seconds() / 3600
dfr_counts = dfr_df.groupby([ACTIVITY_COL, 'next_activity'], dropna=False).size().reset_index(name='n_transitions')
dfr_counts = dfr_counts.sort_values('n_transitions', ascending=False).reset_index(drop=True)
dfr_counts['transition_share_pct'] = dfr_counts['n_transitions'] / len(dfr_df) * 100
save_table(dfr_counts, 'audit_15_directly_follows_counts', show_rows=50)
dfr_nonneg = dfr_df[dfr_df['delta_hours_to_next'].notna() & (dfr_df['delta_hours_to_next'] >= 0)].copy()
dfr_perf = dfr_nonneg.groupby([ACTIVITY_COL, 'next_activity']).agg(n=('delta_hours_to_next', 'size'), median_hours=('delta_hours_to_next', 'median'), mean_hours=('delta_hours_to_next', 'mean'), p90_hours=('delta_hours_to_next', lambda x: np.percentile(x, 90))).reset_index()
dfr_perf = dfr_perf.sort_values(['median_hours', 'n'], ascending=[False, False]).reset_index(drop=True)
save_table(dfr_perf, 'audit_16_directly_follows_performance', show_rows=50)


In [ ]:
# Gleiche Zeitstempel
tie_groups = df_sorted.groupby([CASE_COL, TIME_COL], dropna=False).size().reset_index(name='n_events_same_case_timestamp')
tie_groups_problem = tie_groups[tie_groups['n_events_same_case_timestamp'] > 1].copy()
n_tie_groups = len(tie_groups_problem)
n_events_in_tie_groups = int(tie_groups_problem['n_events_same_case_timestamp'].sum())
n_cases_with_ties = tie_groups_problem[CASE_COL].nunique(dropna=True)
n_cases_total = df_sorted[CASE_COL].nunique(dropna=True)
tie_summary = pd.DataFrame([{'n_case_timestamp_groups_total': len(tie_groups), 'n_tie_groups': n_tie_groups, 'n_events_in_tie_groups': n_events_in_tie_groups, 'pct_events_in_tie_groups': n_events_in_tie_groups / len(df_sorted) * 100, 'n_cases_with_at_least_one_tie': n_cases_with_ties, 'pct_cases_with_at_least_one_tie': n_cases_with_ties / n_cases_total * 100, 'max_events_same_case_timestamp': int(tie_groups['n_events_same_case_timestamp'].max())}])
save_table(tie_summary, 'audit_17_timestamp_tie_summary', show_rows=None)
save_table(tie_groups_problem.sort_values('n_events_same_case_timestamp', ascending=False).head(100), 'audit_18_timestamp_tie_groups_top100', show_rows=30)


In [ ]:
# Beispiele gleicher Zeitstempel
if len(tie_groups_problem) > 0:
    top_tie_keys = tie_groups_problem.sort_values('n_events_same_case_timestamp', ascending=False).head(10)[[CASE_COL, TIME_COL]]
    tie_examples = df_sorted.merge(top_tie_keys, on=[CASE_COL, TIME_COL], how='inner')
    tie_examples = tie_examples.sort_values([CASE_COL, TIME_COL, '__row_order__'])
    save_table(tie_examples[[CASE_COL, TIME_COL, ACTIVITY_COL, '__row_order__']].head(200), 'audit_19_timestamp_tie_examples', show_rows=60)
else:
    print('Keine Timestamp-Ties gefunden.')


In [ ]:
# Duplikate
full_duplicate_count = int(df.drop(columns=['__row_order__'], errors='ignore').duplicated().sum())
cat_duplicate_count = int(df.duplicated(subset=[CASE_COL, ACTIVITY_COL, TIME_COL]).sum())
negative_dfr_count = int((dfr_df['delta_hours_to_next'] < 0).sum())
zero_dfr_count = int((dfr_df['delta_hours_to_next'] == 0).sum())
duplicate_summary = pd.DataFrame([{'full_duplicate_rows_excluding_row_order': full_duplicate_count, 'duplicate_case_activity_timestamp_rows': cat_duplicate_count, 'negative_time_to_next_event_after_sorting': negative_dfr_count, 'zero_time_to_next_event_after_sorting': zero_dfr_count, 'pct_zero_time_to_next_event': zero_dfr_count / len(dfr_df) * 100}])
save_table(duplicate_summary, 'audit_20_duplicate_time_anomaly_summary', show_rows=None)
if cat_duplicate_count > 0:
    dup_examples = df[df.duplicated(subset=[CASE_COL, ACTIVITY_COL, TIME_COL], keep=False)].sort_values([CASE_COL, TIME_COL, ACTIVITY_COL]).head(100)
    save_table(dup_examples[[CASE_COL, ACTIVITY_COL, TIME_COL, '__row_order__']], 'audit_21_case_activity_timestamp_duplicate_examples', show_rows=40)


In [ ]:
# Varianten
variant_series = df_sorted.groupby(CASE_COL)[ACTIVITY_COL].agg(lambda x: ' > '.join(map(str, x)))
variant_counts = variant_series.value_counts().reset_index()
variant_counts.columns = ['variant', 'n_cases']
variant_counts['case_share_pct'] = variant_counts['n_cases'] / n_cases_total * 100
variant_counts['cum_case_share_pct'] = variant_counts['case_share_pct'].cumsum()
variant_counts['variant_rank'] = np.arange(1, len(variant_counts) + 1)
variant_counts['variant_length_events'] = variant_counts['variant'].str.count(' > ') + 1
variant_summary = pd.DataFrame([{'n_variants': len(variant_counts), 'n_cases': n_cases_total, 'top1_variant_case_share_pct': variant_counts.loc[0, 'case_share_pct'] if len(variant_counts) else np.nan, 'top5_variants_case_share_pct': variant_counts.head(5)['case_share_pct'].sum(), 'top10_variants_case_share_pct': variant_counts.head(10)['case_share_pct'].sum(), 'top50_variants_case_share_pct': variant_counts.head(50)['case_share_pct'].sum(), 'variants_covering_80pct_cases': int((variant_counts['cum_case_share_pct'] <= 80).sum() + 1) if len(variant_counts) else 0}])
save_table(variant_summary, 'audit_22_variant_summary', show_rows=None)
save_table(variant_counts.head(200), 'audit_23_top200_variants', show_rows=30)


In [ ]:
# Variantenabdeckung
plt.figure(figsize=(9, 5))
plt.plot(variant_counts['variant_rank'].head(500), variant_counts['cum_case_share_pct'].head(500))
plt.xlabel('Variant Rank')
plt.ylabel('Kumulative Case-Abdeckung in %')
plt.title('Kumulative Abdeckung der häufigsten Varianten')
save_fig('audit_variant_cumulative_coverage_top500')


In [ ]:
# Wiederholungen
case_activity_counts = df_sorted.groupby([CASE_COL, ACTIVITY_COL]).size()
repeated_case_activity = case_activity_counts[case_activity_counts > 1]
cases_with_repeated_activity = repeated_case_activity.index.get_level_values(0).nunique()
rework_summary = pd.DataFrame([{'n_cases_total': n_cases_total, 'n_cases_with_repeated_activity': cases_with_repeated_activity, 'pct_cases_with_repeated_activity': cases_with_repeated_activity / n_cases_total * 100, 'mean_rework_intensity_simple': case_table['rework_intensity_simple'].mean(), 'median_rework_intensity_simple': case_table['rework_intensity_simple'].median(), 'p90_rework_intensity_simple': case_table['rework_intensity_simple'].quantile(0.9), 'p99_rework_intensity_simple': case_table['rework_intensity_simple'].quantile(0.99)}])
save_table(rework_summary, 'audit_24_rework_summary', show_rows=None)
repeated_activity_excess = (repeated_case_activity - 1).groupby(level=1).sum().sort_values(ascending=False).reset_index()
repeated_activity_excess.columns = [ACTIVITY_COL, 'excess_repetitions_over_first_occurrence']
save_table(repeated_activity_excess, 'audit_25_repeated_activity_excess', show_rows=50)
self_loops = dfr_counts[dfr_counts[ACTIVITY_COL] == dfr_counts['next_activity']].copy()
save_table(self_loops, 'audit_26_self_loops', show_rows=50)


In [ ]:
# Wiederholungen abbilden
plt.figure(figsize=(9, 5))
case_table['rework_intensity_simple'].hist(bins=60)
plt.xlabel('n_events - n_unique_activities')
plt.ylabel('Anzahl Cases')
plt.title('Einfache Rework-Intensität pro Case')
save_fig('audit_rework_intensity_hist')
cap_rework = case_table['rework_intensity_simple'].quantile(0.99)
plt.figure(figsize=(9, 5))
case_table.loc[case_table['rework_intensity_simple'] <= cap_rework, 'rework_intensity_simple'].hist(bins=60)
plt.xlabel('Rework-Intensität bis 99%-Quantil')
plt.ylabel('Anzahl Cases')
plt.title('Rework-Intensität ohne oberstes 1%')
save_fig('audit_rework_intensity_hist_p99')


In [ ]:
# Kandidatenspalten
def cols_containing(keywords):
    keywords = [k.lower() for k in keywords]
    return [c for c in df.columns if any((k in c.lower() for k in keywords))]
year_cols = cols_containing(['year', 'jahr'])
department_cols = cols_containing(['department', 'dept', 'org:group', 'org:resource', 'resource', 'group'])
document_cols = cols_containing(['document', 'doc', 'doctype', 'documenttype'])
subprocess_cols = cols_containing(['subprocess', 'sub process', 'sub_process'])
applicant_cols = cols_containing(['applicant', 'antragsteller', 'person', 'customer'])
amount_cols = cols_containing(['amount', 'payment', 'zahl', 'betrag', 'declared', 'applied', 'selected', 'penalty', 'risk'])
candidate_cols = pd.DataFrame({'category': ['year', 'department/resource', 'document', 'subprocess', 'applicant', 'amount/payment/risk'], 'columns_detected': [year_cols, department_cols, document_cols, subprocess_cols, applicant_cols, amount_cols]})
save_table(candidate_cols, 'audit_27_candidate_columns', show_rows=None)


In [ ]:
# Ausprägungen der Kandidatenspalten
important_cols = []
for group in [year_cols, department_cols, document_cols, subprocess_cols, applicant_cols, amount_cols]:
    for c in group:
        if c not in important_cols and c in df.columns and (c != '__row_order__'):
            important_cols.append(c)
print('Kandidatenspalten:', important_cols)
candidate_value_tables = []
for c in important_cols:
    vc = df[c].value_counts(dropna=False).head(30).reset_index()
    vc.columns = ['value', 'n_events']
    vc.insert(0, 'column', c)
    candidate_value_tables.append(vc)
if candidate_value_tables:
    candidate_values = pd.concat(candidate_value_tables, ignore_index=True)
    save_table(candidate_values, 'audit_28_candidate_column_top_values', show_rows=100)
else:
    print('Keine Kandidatenspalten automatisch erkannt.')


In [ ]:
# Wiederkehrende Antragsteller
if applicant_cols:
    applicant_col = applicant_cols[0]
    applicant_case_counts = df_sorted[[CASE_COL, applicant_col]].dropna().drop_duplicates().groupby(applicant_col)[CASE_COL].nunique().sort_values(ascending=False).reset_index()
    applicant_case_counts.columns = [applicant_col, 'n_cases']
    save_table(applicant_case_counts, 'audit_29_applicant_case_counts', show_rows=50)
    if year_cols:
        year_col = year_cols[0]
        applicant_years = df_sorted[[applicant_col, year_col]].dropna().drop_duplicates().groupby(applicant_col)[year_col].nunique().sort_values(ascending=False).reset_index()
        applicant_years.columns = [applicant_col, 'n_years_observed']
        save_table(applicant_years, 'audit_30_applicant_year_counts', show_rows=50)
else:
    print('Keine Antragstellerspalte erkannt.')


In [ ]:
# Merkmalsverfügbarkeit
first_rows = df_sorted.groupby(CASE_COL, sort=False).head(1)
availability_rows = []
for c in df.columns:
    if c == '__row_order__':
        continue
    event_non_null_pct = df[c].notna().mean() * 100
    first_event_non_null_pct = first_rows[c].notna().mean() * 100
    n_unique = df[c].nunique(dropna=True)
    cases_ever_non_null = df.loc[df[c].notna(), CASE_COL].nunique(dropna=True)
    cases_ever_non_null_pct = cases_ever_non_null / n_cases_total * 100
    try:
        case_nunique = df_sorted.groupby(CASE_COL)[c].nunique(dropna=True)
        pct_cases_value_changes = (case_nunique > 1).mean() * 100
    except Exception:
        pct_cases_value_changes = np.nan
    availability_rows.append({'column': c, 'dtype': str(df[c].dtype), 'n_unique': int(n_unique), 'event_non_null_pct': round(event_non_null_pct, 3), 'first_event_non_null_pct': round(first_event_non_null_pct, 3), 'cases_ever_non_null_pct': round(cases_ever_non_null_pct, 3), 'pct_cases_value_changes': round(pct_cases_value_changes, 3) if pd.notna(pct_cases_value_changes) else np.nan, 'preliminary_interpretation': 'prüfen: leakage möglich' if first_event_non_null_pct < 80 and cases_ever_non_null_pct > first_event_non_null_pct + 10 else 'eher früh/verfügbar oder konstant'})
availability = pd.DataFrame(availability_rows).sort_values(['preliminary_interpretation', 'first_event_non_null_pct'], ascending=[False, True])
save_table(availability, 'audit_31_feature_availability_first_event', show_rows=80)


In [ ]:
# Aktivitätsmuster
patterns = ['reopen', 'reopened', 'open', 'close', 'closed', 'payment', 'pay', 'late', 'deadline', 'approve', 'approved', 'reject', 'rejected', 'refuse', 'abort', 'cancel', 'withdraw', 'valid', 'invalid', 'check', 'control', 'parcel', 'geo', 'reference', 'alignment', 'begin', 'start', 'finish', 'end']
activity_names = activity_counts[[ACTIVITY_COL, 'n_events']].copy()
activity_names['activity_lower'] = activity_names[ACTIVITY_COL].astype(str).str.lower()
pattern_rows = []
for pat in patterns:
    matches = activity_names[activity_names['activity_lower'].str.contains(pat, regex=False, na=False)].copy()
    if len(matches) > 0:
        for _, row in matches.head(30).iterrows():
            pattern_rows.append({'pattern': pat, 'activity': row[ACTIVITY_COL], 'n_events': row['n_events']})
pattern_matches = pd.DataFrame(pattern_rows)
if len(pattern_matches):
    save_table(pattern_matches, 'audit_32_activity_pattern_matches', show_rows=120)
else:
    print('Keine passenden Aktivitätsmuster gefunden.')


In [ ]:
# Fallkennzeichen
case_flag_rows = []
for pat in ['reopen', 'payment', 'pay', 'late', 'approve', 'reject', 'abort', 'valid', 'check', 'control']:
    mask = df_sorted[ACTIVITY_COL].astype(str).str.lower().str.contains(pat, regex=False, na=False)
    n_events_pat = int(mask.sum())
    n_cases_pat = df_sorted.loc[mask, CASE_COL].nunique(dropna=True)
    case_flag_rows.append({'pattern': pat, 'n_matching_events': n_events_pat, 'n_matching_cases': n_cases_pat, 'pct_cases': n_cases_pat / n_cases_total * 100})
case_flag_table = pd.DataFrame(case_flag_rows).sort_values('n_matching_cases', ascending=False)
save_table(case_flag_table, 'audit_33_exploratory_case_flags_by_activity_pattern', show_rows=None)


In [ ]:
# Methodische Hinweise
warnings_list = []
if n_cases_with_ties > 0:
    warnings_list.append(f'Timestamp-Ties: {n_cases_with_ties} Cases ({n_cases_with_ties / n_cases_total * 100:.2f}%) haben mindestens einen identischen Timestamp innerhalb desselben Cases. Reihenfolge bei Ties muss dokumentiert werden.')
if full_duplicate_count > 0 or cat_duplicate_count > 0:
    warnings_list.append(f'Dubletten: {full_duplicate_count} vollständige Dubletten und {cat_duplicate_count} Case-Activity-Timestamp-Dubletten gefunden. Bedeutung prüfen.')
if negative_dfr_count > 0:
    warnings_list.append(f'Negative Zeitabstände nach Sortierung: {negative_dfr_count}. Timestamp-Qualität prüfen.')
if case_table['duration_days'].isna().any():
    warnings_list.append('Einige Cases haben keine berechenbare Durchlaufzeit. Missing Timestamps prüfen.')
leakage_candidates = availability[(availability['first_event_non_null_pct'] < 50) & (availability['cases_ever_non_null_pct'] > 80)].copy()
if len(leakage_candidates):
    warnings_list.append(f'Feature Availability: {len(leakage_candidates)} Spalten sind beim ersten Event selten befüllt, aber irgendwann in vielen Cases vorhanden. Für frühe Prediction Leakage prüfen.')
warnings_df = pd.DataFrame({'methodological_warning': warnings_list}) if warnings_list else pd.DataFrame({'methodological_warning': ['Keine automatischen Warnhinweise erzeugt. Trotzdem manuell prüfen.']})
save_table(warnings_df, 'audit_34_methodological_warnings', show_rows=None)
